# Notebook 05 - Sistema Completo com Seguranca e Logging

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Integracao de todos os componentes
2. Sistema de seguranca e validacao
3. Logging detalhado para auditoria
4. Explainability das respostas
5. Demonstracao final do assistente

---
## 1. Configuracao do Ambiente

In [ ]:
import os
import sys
import json
import logging
import torch
from datetime import datetime
from typing import TypedDict, List, Optional
from dotenv import load_dotenv
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline

# Importar modulo RAG do projeto
sys.path.append('..')
from src.rag_module import (
    SimpleDeterministicEmbeddings,
    create_medical_documents,
    create_vector_store,
    create_retriever
)

load_dotenv()

# Configuracao do Modelo Fine-Tuned
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "../models/assistente_medico_final"

print("Sistema Assistente Medico - Versao Completa")
print("="*50)
print(f"Modelo base: {MODEL_NAME}")
print(f"Adapter: {ADAPTER_PATH}")

---
## 2. Sistema de Logging

Logging detalhado e essencial para auditoria e rastreabilidade em sistemas medicos.

In [ ]:
# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('../logs/assistente_medico.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('AssistenteMedico')

def registrar_consulta(pergunta, resposta, fontes, confianca, medico_id="anonimo"):
    """Registra cada consulta para auditoria."""
    registro = {
        "timestamp": datetime.now().isoformat(),
        "medico_id": medico_id,
        "pergunta": pergunta[:200],  # Limitar para privacidade
        "resposta_gerada": resposta[:500],
        "fontes_consultadas": fontes,
        "nivel_confianca": confianca,
        "validacao_humana": False
    }
    
    logger.info("CONSULTA_ASSISTENTE", extra=registro)
    
    # Salvar em JSON para auditoria
    with open('../logs/auditoria.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(registro, ensure_ascii=False) + '\n')
    
    return registro

print("Sistema de logging configurado!")
print("Logs salvos em: ../logs/assistente_medico.log")
print("Auditoria em: ../logs/auditoria.jsonl")

---
## 3. Modulo de Seguranca

### 3.1 Regras de Seguranca

In [ ]:
# Regras inegociaveis do assistente medico
REGRAS_SEGURANCA = {
    "prompt_seguranca": """
    REGRAS INEGOCIAVEIS DO ASSISTENTE MEDICO:
    1. NUNCA prescreva medicamentos diretamente
    2. NUNCA faca diagnosticos definitivos - apenas sugira hipoteses
    3. SEMPRE inclua: 'Esta resposta e uma sugestao gerada por IA e deve ser validada por um medico especialista antes da decisao clinica.'
    4. SEMPRE cite a fonte do protocolo utilizado
    5. Se nao tiver certeza, diga 'Nao tenho informacao suficiente para esta pergunta'
    6. NUNCA compartilhe dados de pacientes identificaveis
    7. SEMPRE recomende consulta com medico especialista
    """,
    "palavras_proibidas": [
        "prescrever", "receitar", "diagnosticar definitivamente",
        "certeza absoluta", "garantia", "100% eficaz"
    ],
    "alertas_urgencia": [
        "emergencia", "urgente", "parada cardiaca", "infarto",
        "derrame", "sangramento", "choque", "parada respiratoria"
    ]
}

def validar_resposta(resposta: str) -> dict:
    """
    Valida se a resposta gerada obedece as regras de seguranca.
    """
    problemas = []
    
    # Verificar palavras proibidas
    for palavra in REGRAS_SEGURANCA["palavras_proibidas"]:
        if palavra.lower() in resposta.lower():
            problemas.append(f"Palavra proibida detectada: '{palavra}'")
    
    # Verificar se contem aviso de seguranca
    aviso_presente = any(
        aviso in resposta.lower()
        for aviso in ["validada por um medico", "sugestao", "especialista"]
    )
    if not aviso_presente:
        problemas.append("Aviso de seguranca nao encontrado na resposta")
    
    # Verificar se contem fontes
    fonte_presente = any(
        fonte in resposta.lower()
        for fonte in ["fonte", "protocolo", "diretriz", "evidencia"]
    )
    if not fonte_presente:
        problemas.append("Referencia a fonte/protocolo nao encontrada")
    
    return {
        "valida": len(problemas) == 0,
        "problemas": problemas,
        "pontuacao_seguranca": max(0, 100 - len(problemas) * 25)
    }

def detectar_urgencia(pergunta: str) -> dict:
    """
    Detecta se a pergunta indica situacao de urgencia.
    """
    pergunta_lower = pergunta.lower()
    
    urgencias_detectadas = []
    for urgencia in REGRAS_SEGURANCA["alertas_urgencia"]:
        if urgencia in pergunta_lower:
            urgencias_detectadas.append(urgencia)
    
    return {
        "e_urgencia": len(urgencias_detectadas) > 0,
        "urgencias": urgencias_detectadas,
        "nivel": "CRITICO" if len(urgencias_detectadas) >= 2 else "ALTO" if urgencias_detectadas else "NORMAL"
    }

print("Modulo de seguranca configurado!")
print(f"Regras ativas: {len(REGRAS_SEGURANCA['palavras_proibidas'])} palavras proibidas")
print(f"Alertas de urgencia: {len(REGRAS_SEGURANCA['alertas_urgencia'])} termos monitorados")

### 3.2 Funcao de Anonimizacao

In [ ]:
import re
import hashlib

def anonimizar_texto(texto: str) -> str:
    """
    Remove dados identificaveis de pessoais (DPIs) do texto.
    Conforme LGPD e regulamentacoes de saude.
    """
    # Mascarar CPF
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF-MASCARADO]', texto)
    # Mascarar RG
    texto = re.sub(r'\d{2}\.\d{3}\.\d{3}-[\dX]', '[RG-MASCARADO]', texto)
    # Mascarar CRMs
    texto = re.sub(r'CRM[/-]?[A-Z]{2}\s*\d+', '[CRM-MASCARADO]', texto)
    # Mascarar nomes comuns
    nomes_comuns = ['Ana', 'Maria', 'Joao', 'Jose', 'Pedro', 'Paula', 'Lucia', 'Fernanda', 'Carlos']
    for nome in nomes_comuns:
        texto = re.sub(rf'\b{nome}\b', '[NOME-MASCARADO]', texto, flags=re.IGNORECASE)
    # Mascarar datas completas
    texto = re.sub(r'\d{2}/\d{2}/\d{4}', '[DATA-MASCARADA]', texto)
    # Mascarar numeros de telefone
    texto = re.sub(r'\(\d{2}\)\s*\d{4,5}-\d{4}', '[TEL-MASCARADO]', texto)
    
    return texto

print("Funcao de anonimizacao configurada!")

---
## 4. Sistema Completo com LangGraph

In [ ]:
from langchain_core.prompts import PromptTemplate

# FLAG: True = usar modelo fine-tuned, False = usar modelo base puro
USE_FINETUNED = True

# AutoTokenizer: detecta automaticamente o tokenizer correto do modelo
# com base no config.json do adapter. Converte texto em numeros (tokens)
# que o LLM processa, e depois converte de volta para texto.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Carregar modelo base (usando float32 para compatibilidade com CPU)
print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map=None,
    trust_remote_code=True
)

# Aplicar adapter LoRA fine-tuned (opcional)
if USE_FINETUNED:
    print("Aplicando adapter LoRA...")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
else:
    print("Usando modelo BASE (sem adapter)")

model.eval()

# Criar pipeline de geracao
pipe = pipeline(
    "text-generation",    # Tipo de tarefa: gerar texto
    model=model,           # Modelo fine-tuned carregado
    tokenizer=tokenizer,   # Tokenizer do adapter
    max_new_tokens=256,    # Maximo de tokens novos a gerar na resposta
    truncation=True,       # Trunca input se for muito longo
    temperature=0.3,       # Criatividade moderada
    do_sample=True,        # Amostragem aleatoria (necessario para temperature > 0)
    top_p=0.9,             # Nucleus sampling: ignora tokens com prob < 10%
    top_k=50,              # Top-K sampling: considera os 50 tokens mais provaveis
    repetition_penalty=1.1,# Penaliza repeticao de tokens (evita loop)
    pad_token_id=tokenizer.eos_token_id  # Token de padding = token de fim
)

# Envolver com LangChain
llm = HuggingFacePipeline(pipeline=pipe)

print(f"Modelo carregado com sucesso!")
print(f"Modo: {'Fine-tuned' if USE_FINETUNED else 'Base'}")

# Configurar RAG (Retrieval-Augmented Generation)
print("\nConfigurando RAG...")
vectorstore = create_vector_store()
retriever = create_retriever(vectorstore, search_k=3)
print(f"Vector store: {vectorstore.index.ntotal} vetores")
print("Retriever configurado!")

# Estado completo do sistema
class AssistenteState(TypedDict):
    pergunta: str
    pergunta_anonimizada: str
    classificacao: str
    nivel_urgencia: str
    documentos: List[Document]
    contexto: str
    resposta_bruta: str
    resposta_validada: str
    fontes: List[str]
    confianca: float
    pontuacao_seguranca: int
    alerta_urgencia: bool
    mensagem_alerta: str
    historico: List[str]
    timestamp: str

# Nos do sistema
def no_anonimizar(state: dict) -> dict:
    """Anonimiza a pergunta antes do processamento."""
    pergunta = state["pergunta"]
    pergunta_anon = anonimizar_texto(pergunta)
    
    historico = state.get("historico", [])
    historico.append("Pergunta anonimizada com sucesso")
    
    return {
        "pergunta_anonimizada": pergunta_anon,
        "historico": historico,
        "timestamp": datetime.now().isoformat()
    }

# ============================================
# NO 2: VERIFICACAO DE URGENCIA
# ============================================
# Objetivo: Detectar palavras-chave de emergencia na pergunta
# Palavras-chave: emergencia, urgente, parada cardiaca, infarto, derrame
# Entrada: pergunta original do medico
# Saida: alerta_urgencia (True/False) + nivel (NORMAL/ALTO/CRITICO)
def no_verificar_urgencia(state: dict) -> dict:
    """Verifica se ha situacao de urgencia."""
    pergunta = state["pergunta"]
    resultado = detectar_urgencia(pergunta)
    
    historico = state.get("historico", [])
    
    if resultado["e_urgencia"]:
        historico.append(f"URGENCIA DETECTADA: {', '.join(resultado['urgencias'])}")
        return {
            "alerta_urgencia": True,
            "mensagem_alerta": f"ALERTA {resultado['nivel']}: Detectada possivel situacao de urgencia. Encaminhe imediatamente para o servico de emergencia.",
            "nivel_urgencia": resultado["nivel"],
            "historico": historico
        }
    else:
        historico.append("Nenhuma urgencia detectada")
        return {
            "alerta_urgencia": False,
            "mensagem_alerta": "",
            "nivel_urgencia": "NORMAL",
            "historico": historico
        }

# ============================================
# NO 3: CLASSIFICACAO DA CONSULTA
# ============================================
# Objetivo: Identificar a especialidade medica da pergunta
# Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
# Entrada: pergunta anonimizada
# Saida: classificacao (nome da categoria)
def no_classificar(state: dict) -> dict:
    """Classifica a consulta medica."""
    pergunta = state["pergunta_anonimizada"]
    
    prompt = PromptTemplate(
        template="""### Instruction:
Classifique: {pergunta}
Categorias: CLINICA_GERAL, CARDIOLOGIA, PNEUMOLOGIA, NEUROLOGIA, URGENCIA
Responda APENAS com a categoria.

### Response:
""",
        input_variables=["pergunta"]
    )
    
    resposta = llm.invoke(prompt.format(pergunta=pergunta))
    classificacao = resposta.strip()
    
    historico = state.get("historico", [])
    historico.append(f"Classificacao: {classificacao}")
    
    return {
        "classificacao": classificacao,
        "historico": historico
    }

# ============================================
# NO 4: RECUPERACAO DE DOCUMENTOS (RAG REAL)
# ============================================
# Objetivo: Buscar protocolos medicos relevantes no vector store FAISS
# Busca os 3 documentos mais similares a pergunta
# Entrada: pergunta anonimizada
# Saida: documentos com protocolos e diretrizes
def no_recuperar(state: dict) -> dict:
    """Recupera documentos relevantes usando RAG real."""
    pergunta = state["pergunta_anonimizada"]
    
    # Buscar documentos relevantes no vector store
    docs = retriever.invoke(pergunta)
    
    historico = state.get("historico", [])
    historico.append(f"RAG: Recuperados {len(docs)} documentos relevantes")
    
    return {
        "documentos": docs,
        "historico": historico
    }

# ============================================
# NO 5: GERACAO DE RESPOSTA
# ============================================
# Objetivo: Gerar a resposta medica estruturada usando o LLM
# Inclui regras de seguranca (nao prescrever, nao diagnosticar)
# Entrada: pergunta, classificacao, documentos RAG
# Saida: resposta com analise, condutas, exames, fontes e aviso
def no_gerar_resposta(state: dict) -> dict:
    """Gera a resposta com seguranca integrada."""
    pergunta = state["pergunta_anonimizada"]
    classificacao = state.get("classificacao", "")
    documentos = state.get("documentos", [])
    
    contexto = "\n".join([doc.page_content for doc in documentos])
    fontes = [doc.metadata.get("fonte", "") for doc in documentos]
    
    prompt = PromptTemplate(
        template="""### Instruction:
{regras}

Contexto: {contexto}

Pergunta: {pergunta}
Classificacao: {classificacao}

Responda de forma estruturada incluindo:
1. Analise clinica
2. Condutas recomendadas
3. Exames complementares
4. Fontes consultadas
5. AVISO DE SEGURANCA OBRIGATORIO

### Response:
""",
        input_variables=["regras", "contexto", "pergunta", "classificacao"]
    )
    
    resposta = llm.invoke(prompt.format(
        regras=REGRAS_SEGURANCA["prompt_seguranca"],
        contexto=contexto,
        pergunta=pergunta,
        classificacao=classificacao
    ))
    
    historico = state.get("historico", [])
    historico.append("Resposta gerada")
    
    return {
        "resposta_bruta": resposta,
        "fontes": fontes,
        "historico": historico
    }

# ============================================
# NO 6: VALIDACAO DE SEGURANCA
# ============================================
# Objetivo: Verificar se a resposta obedece as regras medicas
# Verificacoes: palavras proibidas, aviso de seguranca, fontes
# Se invalida: adiciona aviso de seguranca automaticamente
# Entrada: resposta_bruta gerada pelo LLM
# Saida: resposta_validada + pontuacao_seguranca (0-100)
def no_validar_seguranca(state: dict) -> dict:
    """Valida a resposta gerada contra as regras de seguranca."""
    resposta = state.get("resposta_bruta", "")
    
    validacao = validar_resposta(resposta)
    
    historico = state.get("historico", [])
    
    if validacao["valida"]:
        historico.append(f"Validacao APROVADA (pontuacao: {validacao['pontuacao_seguranca']})")
        return {
            "resposta_validada": resposta,
            "pontuacao_seguranca": validacao["pontuacao_seguranca"],
            "historico": historico
        }
    else:
        # Adicionar aviso se necessario
        aviso = "\n\n[AVISO DE SEGURANCA] Esta resposta deve ser revisada por um medico especialista."
        historico.append(f"Validacao com OBSERVACOES: {', '.join(validacao['problemas'])}")
        return {
            "resposta_validada": resposta + aviso,
            "pontuacao_seguranca": validacao["pontuacao_seguranca"],
            "historico": historico
        }

# ============================================
# NO 7: REGISTRO PARA AUDITORIA
# ============================================
# Objetivo: Salvar a consulta para auditoria e rastreabilidade
# Registra: timestamp, medico_id, pergunta, resposta, fontes, confianca
# Salva em JSON para consultas futuras e conformidade LGPD
# Entrada: todos os dados da consulta
# Saida: registro salvo em logs/auditoria.jsonl
def no_registrar(state: dict) -> dict:
    """Registra a consulta para auditoria."""
    registrar_consulta(
        pergunta=state.get("pergunta", ""),
        resposta=state.get("resposta_validada", ""),
        fontes=state.get("fontes", []),
        confianca=state.get("confianca", 0),
        medico_id="medico_anonimo"
    )
    
    historico = state.get("historico", [])
    historico.append("Consulta registrada para auditoria")
    
    return {"historico": historico}

print("Todos os nos do sistema implementados!")

---
## 5. Construcao do Grafo Final

In [ ]:
# ============================================
# CONSTRUCAO DO GRAFO FINAL
# ============================================
# Fluxo completo com 7 nos:
# 1. Anonimizar (LGPD)
# 2. Verificar Urgencia (palavras-chave)
# 3. Classificar (especialidade)
# 4. Recuperar (RAG real com FAISS)
# 5. Gerar Resposta (LLM + regras)
# 6. Validar Seguranca (conformidade)
# 7. Registrar (auditoria)

workflow = StateGraph(AssistenteState)

# Adicionar nos do grafo
workflow.add_node("anonimizar", no_anonimizar)             # No 1
workflow.add_node("verificar_urgencia", no_verificar_urgencia)  # No 2
workflow.add_node("classificar", no_classificar)           # No 3
workflow.add_node("recuperar", no_recuperar)               # No 4 (RAG)
workflow.add_node("gerar_resposta", no_gerar_resposta)     # No 5
workflow.add_node("validar_seguranca", no_validar_seguranca)  # No 6
workflow.add_node("registrar", no_registrar)               # No 7

# Fluxo principal (sequencial)
workflow.set_entry_point("anonimizar")
workflow.add_edge("anonimizar", "verificar_urgencia")
workflow.add_edge("verificar_urgencia", "classificar")
workflow.add_edge("classificar", "recuperar")
workflow.add_edge("recuperar", "gerar_resposta")
workflow.add_edge("gerar_resposta", "validar_seguranca")
workflow.add_edge("validar_seguranca", "registrar")
workflow.add_edge("registrar", END)

# Compilar o grafo
assistente = workflow.compile()

print("Sistema Assistente Medico compilado!")
print("\nFluxo:")
print("Anonimizar -> Verificar Urgencia -> Classificar -> Recuperar (RAG) ->")
print("Gerar Resposta -> Validar Seguranca -> Registrar -> Fim")

---
## 6. Demonstracao do Sistema

In [ ]:
# Funcao para executar o assistente
def executar_assistente(pergunta: str, medico_id: str = "anonimo") -> dict:
    """
    Executa o assistente medico completo.
    """
    print(f"\n{'='*60}")
    print("EXECUTANDO ASSISTENTE MEDICO")
    print("="*60)
    print(f"Pergunta: {pergunta}")
    print(f"Medico ID: {medico_id}")
    print("-"*60)
    
    entrada = {
        "pergunta": pergunta,
        "historico": []
    }
    
    resultado = assistente.invoke(entrada)
    
    # Exibir resultados
    print(f"\nClassificacao: {resultado['classificacao']}")
    print(f"Nivel urgencia: {resultado['nivel_urgencia']}")
    print(f"Pontuacao seguranca: {resultado['pontuacao_seguranca']}/100")
    print(f"Fontes: {resultado['fontes']}")
    
    if resultado['alerta_urgencia']:
        print(f"\n{'!'*60}")
        print(f"ALERTA: {resultado['mensagem_alerta']}")
        print(f"{'!'*60}")
    
    print(f"\nHistorico:")
    for i, h in enumerate(resultado['historico'], 1):
        print(f"  {i}. {h}")
    
    print(f"\n{'='*60}")
    print("RESPOSTA VALIDADA:")
    print("="*60)
    print(resultado['resposta_validada'])
    
    return resultado

print("Funcao de demonstracao pronta!")

In [ ]:
# DEMONSTRACAO 1: Consulta normal
resultado1 = executar_assistente(
    "Qual o protocolo para pneumonia hospitalar em pacientes idosos?",
    medico_id="dr_silva"
)

In [ ]:
# DEMONSTRACAO 2: Situacao de urgencia
resultado2 = executar_assistente(
    "Paciente com emergencia de infarto agudo do miocardio. Dor toracica intensa.",
    medico_id="dr_santos"
)

In [ ]:
# DEMONSTRACAO 3: Consulta com dados sensiveis (testar anonimizacao)
resultado3 = executar_assistente(
    "Paciente Maria Silva, CPF 123.456.789-00, com quadro de sepse. Qual conduta?",
    medico_id="dr橄榄" # Aqui ja passamos um ID, entao sera usado
)

---
## 7. Explainability (Explicabilidade)

A explicabilidade e essencial em sistemas medicos para:
- Indicar fonte de cada informacao
- Mostrar nivel de confianca
- Registrar todas as etapas
- Permitir auditoria completa

In [ ]:
def gerar_relatorio_explainability(resultado: dict) -> str:
    """
    Gera relatorio de explicabilidade da resposta.
    """
    relatorio = f"""
    RELATORIO DE EXPLICABILIDADE
    ============================
    
    Data/Hora: {resultado.get('timestamp', 'N/A')}
    
    1. CLASSIFICACAO:
       Categoria: {resultado.get('classificacao', 'N/A')}
       
    2. URGENCIA:
       Nivel: {resultado.get('nivel_urgencia', 'N/A')}
       Alerta: {'SIM' if resultado.get('alerta_urgencia') else 'NAO'}
       
    3. FONTES CONSULTADAS:
       {chr(10).join(f'- {f}' for f in resultado.get('fontes', []))}
    
    4. SEGURANCA:
       Pontuacao: {resultado.get('pontuacao_seguranca', 0)}/100
       
    5. HISTORICO DE DECISOES:
       {chr(10).join(f'   {i}. {h}' for i, h in enumerate(resultado.get('historico', []), 1))}
    
    6. AVISO LEGAL:
       Esta resposta foi gerada por Inteligencia Artificial.
       Todas as informacoes devem ser validadas por um medico
       especialista antes da decisao clinica.
    
    7. CONFORMIDADE:
       - LGPD: Dados anonimizados
       - CFF: Respeita etica medica
       - ANVISA: Seguranca de software medico
    """
    
    return relatorio

# Gerar relatorio para a ultima consulta
if resultado3:
    print(gerar_relatorio_explainability(resultado3))

---
## 8. Metricas e Monitoramento

In [ ]:
# Carregar e analisar logs de auditoria
import os

log_file = '../logs/auditoria.jsonl'

if os.path.exists(log_file):
    registros = []
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                registros.append(json.loads(line))
    
    print(f"Total de consultas registradas: {len(registros)}")
    
    # Analise basica
    if registros:
        confiancas = [r.get('nivel_confianca', 0) for r in registros]
        print(f"Confianca media: {sum(confiancas)/len(confiancas):.2%}")
        print(f"\nUltimas consultas:")
        for r in registros[-3:]:
            print(f"  - {r.get('timestamp', 'N/A')}: {r.get('pergunta', 'N/A')[:50]}...")
else:
    print("Nenhum log de auditoria encontrado ainda.")
    print("Execute algumas consultas primeiro.")

---
## 9. Resumo Final

### Sistema Implementado:

| Componente | Descricao |
|------------|----------|
| **Fine-Tuning** | Modelo treinado com dados medicos (Notebook 02) |
| **LangChain** | Prompts, Chains, Loaders, Agents (Notebook 03) |
| **LangGraph** | Grafo de decisao com fluxos condicionais (Notebook 04) |
| **RAG** | Recuperacao de documentos + Geracao (Notebook 04) |
| **Seguranca** | Validacao, anonimizacao, regras medicas |
| **Logging** | Auditoria detalhada de todas as consultas |
| **Explainability** | Transparencia total nas decisoes |

### Fluxo Completo:
```
Pergunta -> Anonimizacao -> Verificacao Urgencia -> Classificacao ->
Recuperacao (RAG) -> Geracao -> Validacao Seguranca -> Registro -> Resposta
```

### Entregaveis:
- 5 notebooks Jupyter com documentacao completa
- Codigo modular e reutilizavel
- Sistema de seguranca implementado
- Logging para auditoria
- Dados sinteticos para demonstracao

### Proximos passos:
- Conectar a datasets reais (PubMedQA, MedQuAD)
- Implementar vector store com FAISS/Pinecone
- Treinar modelo com mais dados
- Deploy em ambiente de producao
- Criar video demonstrativo (15 min)

In [ ]:
print("\n" + "="*60)
print("ASSISTENTE MEDICO - SISTEMA COMPLETO")
print("="*60)
print("Tech Challenge Fase 3 - FIAP")
print("Projeto: Assistente Virtual Medico")
print("Tecnologias: LangChain, LangGraph, Fine-Tuning, RAG")
print("="*60)
print("\nSistema pronto para demonstracao!")
print("Execute as celulas de demonstracao para testar o assistente.")